# 04 - Prédiction des 12 prochains mois
## Objectif : Prédire le CA de Fév 2026 → Jan 2027

In [68]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import joblib
import json
import clickhouse_connect
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:,.2f}'.format)
print("✅ Imports OK")

✅ Imports OK


In [69]:
import joblib
import json
import pandas as pd
import numpy as np

# Charger le modèle et le scaler
model  = joblib.load('/app/notebooks/best_model.pkl')
scaler = joblib.load('/app/notebooks/scaler.pkl')

# Charger les features
with open('/app/notebooks/features_corr.json', 'r') as f:
    content = f.read()
    print(f"Contenu JSON features : {content}")
    FEATURES = json.loads(content)

# Charger le dataset
df_ml = pd.read_csv('/app/notebooks/ml_dataset_corr.csv', parse_dates=['date'])

TARGET = 'total_sales'

print(f"\n✅ Modèle chargé : {type(model).__name__}")
print(f"✅ Features ({len(FEATURES)}) : {FEATURES}")
print(f"✅ Dernier mois connu : {df_ml['date'].max().strftime('%b %Y')}")

Contenu JSON features : ["year", "lag_1", "semester", "rolling_mean_3", "month", "quarter", "lag_12"]

✅ Modèle chargé : Ridge
✅ Features (7) : ['year', 'lag_1', 'semester', 'rolling_mean_3', 'month', 'quarter', 'lag_12']
✅ Dernier mois connu : Jan 2026


## 1. Prédiction itérative des 12 prochains mois

In [70]:
# On part du dataset complet pour avoir accès aux valeurs passées
df_future = df_ml.copy()

# Dernier mois connu
last_date = df_future['date'].max()
print(f"Départ : {last_date.strftime('%b %Y')}")
print(f"Objectif : prédire jusqu'à {(last_date + pd.DateOffset(months=12)).strftime('%b %Y')}\n")

predictions = []

for i in range(1, 13):
    next_date  = last_date + pd.DateOffset(months=i)
    next_month = next_date.month
    next_year  = next_date.year

    row = {}

    # ── Features temporelles ──────────────────────────────
    if 'year'        in FEATURES: row['year']        = next_year
    if 'month'       in FEATURES: row['month']       = next_month
    if 'quarter'     in FEATURES: row['quarter']     = (next_month - 1) // 3 + 1
    if 'semester'    in FEATURES: row['semester']    = 1 if next_month <= 6 else 2
    if 'is_summer'   in FEATURES: row['is_summer']   = 1 if next_month in [7, 8] else 0
    if 'is_end_year' in FEATURES: row['is_end_year'] = 1 if next_month in [11, 12] else 0
    if 'is_january'  in FEATURES: row['is_january']  = 1 if next_month == 1 else 0

    # ── Lag features ─────────────────────────────────────
    if 'lag_1'  in FEATURES: row['lag_1']  = all_sales[-1]
    if 'lag_2'  in FEATURES: row['lag_2']  = all_sales[-2] if len(all_sales) >= 2  else all_sales[-1]
    if 'lag_3'  in FEATURES: row['lag_3']  = all_sales[-3] if len(all_sales) >= 3  else all_sales[-1]
    if 'lag_6'  in FEATURES: row['lag_6']  = all_sales[-6] if len(all_sales) >= 6  else all_sales[-1]
    if 'lag_12' in FEATURES:
        past_same_month = df_future[df_future['month'] == next_month]['total_sales']
        row['lag_12'] = past_same_month.iloc[-1] if len(past_same_month) > 0 else all_sales[-1]

    # ── Rolling features ──────────────────────────────────
    if 'rolling_mean_3' in FEATURES: row['rolling_mean_3'] = np.mean(all_sales[-3:])
    if 'rolling_mean_6' in FEATURES: row['rolling_mean_6'] = np.mean(all_sales[-6:])
    if 'rolling_std_3'  in FEATURES: row['rolling_std_3']  = np.std(all_sales[-3:])  if len(all_sales) >= 3 else 0
    if 'rolling_max_3'  in FEATURES: row['rolling_max_3']  = np.max(all_sales[-3:])
    if 'rolling_min_3'  in FEATURES: row['rolling_min_3']  = np.min(all_sales[-3:])
    # ── Features métier — estimées par moyenne des 3 derniers mois ────
    # On suppose que le comportement futur ≈ comportement récent passé
    
    if 'total_quantity' in FEATURES:
        row['total_quantity'] = df_future['total_quantity'].tail(3).mean()
    
    if 'nb_documents' in FEATURES:
        row['nb_documents'] = df_future['nb_documents'].tail(3).mean()
    
    if 'nb_clients' in FEATURES:
        row['nb_clients'] = df_future['nb_clients'].tail(3).mean()
    
    if 'nb_produits' in FEATURES:
        row['nb_produits'] = df_future['nb_produits'].tail(3).mean()
    
    if 'nb_regions' in FEATURES:
        row['nb_regions'] = df_future['nb_regions'].tail(3).mean()
    
    if 'nb_warehouses' in FEATURES:
        row['nb_warehouses'] = df_future['nb_warehouses'].tail(3).mean()
    
    if 'avg_discount' in FEATURES:
        row['avg_discount'] = df_future['avg_discount'].tail(3).mean()
    
    if 'total_discount' in FEATURES:
        row['total_discount'] = df_future['total_discount'].tail(3).mean()
    # ── Prédiction ────────────────────────────────────────
    X_row    = pd.DataFrame([row])[FEATURES]
    X_row_sc = scaler.transform(X_row) if hasattr(scaler, 'mean_') else X_row
    pred     = model.predict(X_row_sc)[0]

    all_sales.append(pred)

    predictions.append({
        'date':            next_date,
        'year':            next_year,
        'month':           next_month,
        'predicted_sales': pred,
        'is_prediction':   1
    })
    print(f"  {next_date.strftime('%b %Y')} → {pred:>15,.2f} DT")

df_pred = pd.DataFrame(predictions)
print(f"\n✅ {len(df_pred)} mois prédits")
print(f"CA prévisionnel total : {df_pred['predicted_sales'].sum():,.2f} DT")

Départ : Jan 2026
Objectif : prédire jusqu'à Jan 2027

  Feb 2026 →    1,276,629.40 DT
  Mar 2026 →    1,238,539.04 DT
  Apr 2026 →    1,304,434.27 DT
  May 2026 →    1,316,360.89 DT
  Jun 2026 →    1,358,031.43 DT
  Jul 2026 →    1,497,039.33 DT
  Aug 2026 →    1,466,143.05 DT
  Sep 2026 →    1,512,297.32 DT
  Oct 2026 →    1,573,215.29 DT
  Nov 2026 →    1,544,605.39 DT
  Dec 2026 →    1,594,143.37 DT
  Jan 2027 →    1,323,330.08 DT

✅ 12 mois prédits
CA prévisionnel total : 17,004,768.87 DT


In [71]:
print(f"FEATURES chargées : {FEATURES}")

FEATURES chargées : ['year', 'lag_1', 'semester', 'rolling_mean_3', 'month', 'quarter', 'lag_12']
